In [1]:
import spatialdata as sd
import spatialdata_plot
import matplotlib.pyplot as plt
import numpy as np
import os
from skimage import io

/home/stefano/miniconda3/envs/analysis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_path = "/home/stefano/Documents/Spatial-Transcriptomic/data/blocco1_sham"
sdata = sd.read_zarr(data_path)
sdata

/tmp/ipykernel_45840/1828072172.py:2: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(data_path)
/home/stefano/miniconda3/envs/analysis/lib/python3.11/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at zmetadata is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


SpatialData object, with associated Zarr store: /home/stefano/Documents/Spatial-Transcriptomic/data/blocco1_sham
├── Images
│     ├── 'blocco1_hires_image': DataArray[cyx] (3, 1849, 4270)
│     ├── 'blocco1_lowres_image': DataArray[cyx] (3, 185, 427)
│     └── 'fluo_image': DataTree[cyx] (3, 7000, 16166), (3, 3500, 8083), (3, 1750, 4041), (3, 875, 2020)
├── Shapes
│     ├── 'GFP_poly': GeoDataFrame shape: (2, 5) (2D shapes)
│     ├── 'blocco1_square_008um': GeoDataFrame shape: (171116, 2) (2D shapes)
│     ├── 'blocco1_square_016um': GeoDataFrame shape: (44363, 1) (2D shapes)
│     ├── 'intissue_008um': GeoDataFrame shape: (95232, 3) (2D shapes)
│     └── 'intissue_poly': GeoDataFrame shape: (1, 5) (2D shapes)
└── Tables
      ├── 'filtered': AnnData (95232, 32285)
      ├── 'final_table': AnnData (95514, 32285)
      ├── 'square_008um': AnnData (95514, 32285)
      └── 'square_016um': AnnData (44363, 32285)
with coordinate systems:
    ▸ 'blocco1', with elements:
        blocco1_hires

In [3]:
from skimage import io as skio

In [4]:
mask= skio.imread("/home/stefano/Downloads/blocco1_sham_masks.tif")

In [18]:
bin_shapes = sdata.shapes['intissue_008um']
#Estrai le coordinate pixel REALI dei bin sfruttando i centroidi della geometria
# Questo estrae i punti fisici (X, Y) nello spazio reale dell'immagine
centroids = bin_shapes.geometry.centroid
x_pixel = centroids.x.values.astype(int)
y_pixel = centroids.y.values.astype(int)

In [28]:
#see how many pixels a bins cover
#for the x_pixel
for i in range(len(x_pixel[:10])):
    if i != 0:
        diff = x_pixel[i] - x_pixel[i-1]
        print(diff)

-25
-25
-26
-25
-25
-26
-25
-25
-26


In [ ]:
print(f"Dimensioni Maschera Cellpose: Altezza (Y) = {mask.shape[0]}, Larghezza (X) = {mask.shape[1]}")
print(f"Range X dei centroidi: min = {x_pixel.min()}, max = {x_pixel.max()}")
print(f"Range Y dei centroidi: min = {y_pixel.min()}, max = {y_pixel.max()}")
#As we can see cellpose mask start from 0,0 coordinate, x_pixels and y_pixels no we need to fix it
x_global = centroids.x.values
y_global = centroids.y.values
mask_height, mask_width = mask.shape  # Prende automaticamente 7000 e 16168

x_pixel = (x_global - x_global.min()) * (mask_width / (x_global.max() - x_global.min() + 1))
y_pixel = (y_global - y_global.min()) * (mask_height / (y_global.max() - y_global.min() + 1))
x_pixel = x_pixel.astype(int)
y_pixel = y_pixel.astype(int)

Dimensioni Maschera Cellpose: Altezza (Y) = 7000, Larghezza (X) = 16168
Range X dei centroidi: min = 647, max = 14853
Range Y dei centroidi: min = 7987, max = 14943


In [ ]:
x_global = centroids.x.values
y_global = centroids.y.values
mask_height, mask_width = mask.shape  # Prende automaticamente 7000 e 16168

#Applica la normalizzazione spaziale 
# Sottraiamo il minimo per azzerare l'offset e scaliammo sulle dimensioni della maschera

x_pixel = (x_global - x_global.min()) * (mask_width / (x_global.max() - x_global.min() + 1))
y_pixel = (y_global - y_global.min()) * (mask_height / (y_global.max() - y_global.min() + 1))
x_pixel = x_pixel.astype(int)
y_pixel = y_pixel.astype(int)

print(f"the area of the image covered by x_pixel x y_pixel is {x_pixel.max()} x {y_pixel.max()}")
#as we can see is very similar to the dimension of the cellpose mask

the area of the image covered by x_pixel x y_pixel is 16166 x 6998


In [52]:
#tabella delle annotazioni con info trascrizionali
adata_bins = sdata.tables['filtered']
print(adata_bins)

# Verifica di sicurezza: il numero di geometrie deve coincidere con le righe dell'AnnData
assert len(bin_shapes) == adata_bins.n_obs, "Discrepanza tra il numero di shapes e i bin"

AnnData object with n_obs × n_vars = 95232 × 32285
    obs: 'in_tissue', 'array_row', 'array_col', 'location_id', 'region', 'sample_id', 'in_treatment', 'GFP_value'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'


In [ ]:
#Mappatura dei bin sulla maschera di Cellpose
fiber_assignments = np.zeros(adata_bins.n_obs, dtype=int)

#Filtro di sicurezza per evitare indici fuori scala (es. clipping sui bordi)
valid_coords = (x_pixel >= 0) & (x_pixel < mask_width) & (y_pixel >= 0) & (y_pixel < mask_height)

#Assegna l'ID della fibra basandoti sulla coordinata pixel corretta (Y=righe, X=colonne)
fiber_assignments[valid_coords] = mask[y_pixel[valid_coords], x_pixel[valid_coords]]

#Salva l'informazione nell'AnnData originale così non perdi il tracciamento
adata_bins.obs['assigned_fiber'] = fiber_assignments

In [105]:
x_pixel

array([8468, 8439, 8410, ..., 5827, 5799, 5770], shape=(95232,))

In [ ]:
#Isola i bin che sono caduti dentro una fibra segmentata (ID > 0)
adata_inside_fibers = adata_bins[adata_bins.obs['assigned_fiber'] > 0]

print(f"Bins totali nel dataset: {len(adata_bins)}")
print(f"Bins mappati dentro le fibre muscolari: {len(adata_inside_fibers)}")

Bin totali nel dataset: 95232
Bin mappati dentro le fibre muscolari: 52162


In [ ]:
#see how many bins are present in the fiber 47
adata_inside_fibers.obs[adata_inside_fibers.obs['assigned_fiber']==47]

,in_tissue,array_row,array_col,location_id,region,sample_id,in_treatment,GFP_value,assigned_fiber
s_008um_00365_00456-1,1,365,456,293383,intissue_008um,sham,False,6.0,47
s_008um_00365_00457-1,1,365,457,293384,intissue_008um,sham,False,6.0,47
s_008um_00365_00458-1,1,365,458,293385,intissue_008um,sham,False,5.0,47
s_008um_00365_00459-1,1,365,459,293386,intissue_008um,sham,False,4.0,47
s_008um_00366_00456-1,1,366,456,294163,intissue_008um,sham,False,12.0,47
s_008um_00366_00457-1,1,366,457,294164,intissue_008um,sham,False,12.0,47
s_008um_00366_00458-1,1,366,458,294165,intissue_008um,sham,False,12.0,47
s_008um_00366_00459-1,1,366,459,294166,intissue_008um,sham,False,12.0,47
s_008um_00366_00460-1,1,366,460,294167,intissue_008um,sham,False,12.0,47
s_008um_00367_00456-1,1,367,456,294957,intissue_008um,sham,False,13.0,47


In [108]:
sdata['filtered']

AnnData object with n_obs × n_vars = 95232 × 32285
    obs: 'in_tissue', 'array_row', 'array_col', 'location_id', 'region', 'sample_id', 'in_treatment', 'GFP_value', 'assigned_fiber'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

Now we verify if the assigned bins to a fiber correspond in terms of coordinate in the cellpose mask

In [ ]:
assigned_indices = np.where(sdata.tables["filtered"].obs['assigned_fiber'] == 47)[0]


In [98]:
sdata.tables['filtered'].obs

,in_tissue,array_row,array_col,location_id,region,sample_id,in_treatment,GFP_value,assigned_fiber
s_008um_00330_00426-1,1,330,426,266106,intissue_008um,sham,False,0.0,0
s_008um_00330_00427-1,1,330,427,266107,intissue_008um,sham,False,0.0,0
s_008um_00330_00428-1,1,330,428,266108,intissue_008um,sham,False,0.0,0
s_008um_00330_00429-1,1,330,429,266109,intissue_008um,sham,False,0.0,0
s_008um_00330_00430-1,1,330,430,266110,intissue_008um,sham,False,0.0,0
...,...,...,...,...,...,...,...,...,...
s_008um_00603_00507-1,1,603,507,477156,intissue_008um,sham,False,3.0,0
s_008um_00603_00508-1,1,603,508,477157,intissue_008um,sham,False,3.0,0
s_008um_00603_00509-1,1,603,509,477158,intissue_008um,sham,False,3.0,0
s_008um_00603_00510-1,1,603,510,477159,intissue_008um,sham,False,4.0,0


Verify if bins are correctly mapped in terms of spatial coordinates inside the segmentates fibers

In [ ]:
# 1. Definiamo la normalizzazione corretta (quella che ha funzionato per l'aggregazione)
mask_height, mask_width = mask.shape

# Ricalcoliamo i pixel normalizzati partendo dai centroidi globali nativi
x_pixel_normalized = (x_global - x_global.min()) * (mask_width / (x_global.max() - x_global.min() + 1))
y_pixel_normalized = (y_global - y_global.min()) * (mask_height / (y_global.max() - y_global.min() + 1))

x_pixel_normalized = x_pixel_normalized.astype(int)
y_pixel_normalized = y_pixel_normalized.astype(int)


# 2. Scegliamo la fibra per il test (es. la fibra 49)
test_fiber_id = 47

# 3. Trova dove si trova la fibra nella maschera di Cellpose
y_indices, x_indices = np.where(mask == test_fiber_id)

if len(x_indices) == 0:
    raise ValueError(f"La fibra {test_fiber_id} non è presente nella maschera. Scegli un altro ID.")

print(f"--- [MASCHERA CELLPOSE] Fibra {test_fiber_id} ---")
print(f"La fibra occupa {len(x_indices)} pixel nella maschera.")
print(f"Bounding Box nei pixel dell'immagine: X=[{x_indices.min()}, {x_indices.max()}], Y=[{y_indices.min()}, {y_indices.max()}]")


# 4. Estraiamo i bin che la pipeline ha assegnato alla test_fiber_id

# Troviamo gli indici posizionali corrispondenti nell'oggetto originario per pescare i pixel corretti
assigned_indices = np.where(sdata.tables["filtered"].obs['assigned_fiber'] == test_fiber_id)[0]

# Prendiamo i pixel normalizzati associati a questi bin
fiber_bins_x = x_pixel_normalized[assigned_indices]
fiber_bins_y = y_pixel_normalized[assigned_indices]

print(f"\n--- [PIPELINE BINS NORMALIZZATI] Fibra {test_fiber_id} ---")
print(f"Numero di bin di Visium HD mappati dentro la fibra: {len(fiber_bins_x)}")


# 5. IL CHECK CRUCIALE
# Verifichiamo se ogni pixel (X, Y) normalizzato del bin cade su un pixel della maschera che vale 49
correct_mappings = 0
incorrect_mappings = 0

for x, y in zip(fiber_bins_x, fiber_bins_y):
    # Controllo di sicurezza per evitare indici fuori matrice
    if 0 <= x < mask_width and 0 <= y < mask_height:
        if mask[y, x] == test_fiber_id:
            correct_mappings += 1
        else:
            incorrect_mappings += 1
    else:
        incorrect_mappings += 1

print(f"\n--- REQUISITI DI VALIDAZIONE ---")
print(f"Bin posizionati correttamente: {correct_mappings}")
print(f"Bin fuori posizione: {incorrect_mappings}")

if incorrect_mappings == 0 and correct_mappings > 0:
    print(f"\n✅ VALIDAZIONE RIUSCITA: Il 100% dei bin estratti cade esattamente sopra la fibra {test_fiber_id}.")
else:
    print(f"\n⚠️ Sfasamento rilevato. Verifica se i bin corretti sono vicini ai bordi della fibra o totalmente disallineati.")

--- [MASCHERA CELLPOSE] Fibra 47 ---
La fibra occupa 14080 pixel nella maschera.
Bounding Box nei pixel dell'immagine: X=[7452, 7591], Y=[852, 999]

--- [PIPELINE BINS NORMALIZZATI] Fibra 47 ---
Numero di bin di Visium HD mappati dentro la fibra: 17

--- REQUISITI DI VALIDAZIONE ---
Bin posizionati correttamente: 17
Bin fuori posizione: 0

✅ VALIDAZIONE RIUSCITA: Il 100% dei bin estratti cade esattamente sopra la fibra 47.
